<a href="https://colab.research.google.com/github/fothstatshs-design/PROJECTS/blob/main/STOCK_PICKS_VS_S%26P500.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# ===================================================
# LIVE STOCK GROWTH ANALYZER (FINAL ROBUST VERSION)
# ===================================================

# --- 1. INSTALL AND IMPORT LIBRARIES ---
# This ensures yfinance is installed in your Colab environment
!pip install yfinance

import pandas as pd
import altair as alt
import yfinance as yf
from IPython.display import display
from datetime import datetime, timedelta

# --- Configuration ---
TODAY = datetime.now().date()
# Calculate historical start dates (using 365.25 days for better leap year approximation)
DATE_3YR_AGO = TODAY - timedelta(days=3 * 365.25)
DATE_5YR_AGO = TODAY - timedelta(days=5 * 365.25)

# Tickers to analyze. ^GSPC is the S&P 500 benchmark.
TICKERS = ['GOOGL', 'META', 'MSFT', 'AMZN', 'UBER', 'CRM', '^GSPC']


# --- 2. DATA FETCHING AND CALCULATION FUNCTION (FIXED AMBIGUITY) ---

def fetch_and_calculate_growth():
    """Fetches live stock data and calculates 3-year and 5-year total growth percentages."""
    growth_3yr = []
    growth_5yr = []

    print(f"Calculating growth using current date: {TODAY.strftime('%Y-%m-%d')}")
    print(f"3-year start date: {DATE_3YR_AGO.strftime('%Y-%m-%d')}")
    print(f"5-year start date: {DATE_5YR_AGO.strftime('%Y-%m-%d')}\n")

    for ticker in TICKERS:
        # Fetch historical data
        stock_data = yf.download(ticker, start=DATE_5YR_AGO, end=TODAY, progress=False, auto_adjust=True)

        if stock_data.empty:
            print(f"Warning: Could not fetch data for {ticker}. Skipping.")
            continue

        current_price = stock_data['Close'].iloc[-1]
        display_ticker = 'S&P 500' if ticker == '^GSPC' else ticker

        # --- Helper function to find historical price (Using .loc for robust slicing) ---
        def get_historical_price_safe(start_date):
            # Use .loc slicing with the date string up to (and including) the start date
            # This avoids the ambiguous boolean Series comparison.
            try:
                # Slice the data up to the desired date and take the last available closing price
                price_series = stock_data.loc[:str(start_date), 'Close']
                if not price_series.empty:
                    return price_series.iloc[-1]
                else:
                    return None # Explicitly return None if the series is empty
            except Exception:
                # Handle cases where the start date is before the first data point
                return None

        # --- 3-Year Calculation ---
        price_3yr_ago = get_historical_price_safe(DATE_3YR_AGO)
        if price_3yr_ago is not None: # Check if the returned value is not None
            # Extract numerical value from Series before calculation
            growth_pct_3yr = ((current_price.iloc[0] / price_3yr_ago.iloc[0]) - 1) * 100
            # Append only the numerical value
            growth_3yr.append({'Ticker': display_ticker, 'Growth_Pct': round(growth_pct_3yr, 2)})
        else:
            print(f"Warning: Not enough historical data for 3-year calculation for {ticker}.")

        # --- 5-Year Calculation ---
        price_5yr_ago = get_historical_price_safe(DATE_5YR_AGO)
        if price_5yr_ago is not None: # Check if the returned value is not None
            # Extract numerical value from Series before calculation
            growth_pct_5yr = ((current_price.iloc[0] / price_5yr_ago.iloc[0]) - 1) * 100
            # Append only the numerical value
            growth_5yr.append({'Ticker': display_ticker, 'Growth_Pct': round(growth_pct_5yr, 2)})
        else:
            print(f"Warning: Not enough historical data for 5-year calculation for {ticker}.")

    df_3yr = pd.DataFrame(growth_3yr)
    df_5yr = pd.DataFrame(growth_5yr)

    # --- ROBUST SORTING FIX ---
    def sort_and_separate_benchmark(df):
        if df.empty or 'S&P 500' not in df['Ticker'].values:
            # Ensure Growth_Pct is numeric before sorting
            df['Growth_Pct'] = pd.to_numeric(df['Growth_Pct'], errors='coerce')
            return df.sort_values(by='Growth_Pct', ascending=False).reset_index(drop=True)

        df_benchmark = df[df['Ticker'] == 'S&P 500'].copy() # Use .copy() to avoid SettingWithCopyWarning
        df_others = df[df['Ticker'] != 'S&P 500'].copy() # Use .copy() to avoid SettingWithCopyWarning

        # Ensure Growth_Pct is numeric before sorting for both dataframes
        df_benchmark['Growth_Pct'] = pd.to_numeric(df_benchmark['Growth_Pct'], errors='coerce')
        df_others['Growth_Pct'] = pd.to_numeric(df_others['Growth_Pct'], errors='coerce')


        df_others = df_others.sort_values(by='Growth_Pct', ascending=False)

        return pd.concat([df_others, df_benchmark]).reset_index(drop=True)

    df_3yr = sort_and_separate_benchmark(df_3yr)
    df_5yr = sort_and_separate_benchmark(df_5yr)


    return df_3yr, df_5yr

# --- 3. CHART CREATION FUNCTION (Requires Altair setup) ---
# Assuming Altair is already imported

def create_growth_chart(df, years, color):
    """Creates an Altair bar chart for stock growth."""
    if df.empty:
        print(f"No data to chart for {years} years.")
        return None

    chart = alt.Chart(df).mark_bar().encode(
        x=alt.X('Ticker', sort='-y'),  # Sort bars by Growth_Pct descending
        y=alt.Y('Growth_Pct', title=f'{years}-Year Growth (%)'),
        tooltip=['Ticker', alt.Tooltip('Growth_Pct', title='Growth (%)')]
    ).properties(
        title=f'Stock Growth Comparison ({years} Years)',
        width=600
    ).interactive()  # Make the chart interactive (zoom and pan)

    return chart


# --- 4. MAIN EXECUTION ---

# Fetch the LIVE dataframes
df_3yr_live, df_5yr_live = fetch_and_calculate_growth()

# Display the 5-year dataframe to check CRM's value
display(df_5yr_live)

# 3-Year Chart
chart_3yr = create_growth_chart(df_3yr_live, 3, 'darkgreen')
if chart_3yr:
    chart_3yr.save('3_year_growth_chart_live.html')
    display(chart_3yr)

# 5-Year Chart
chart_5yr = create_growth_chart(df_5yr_live, 5, 'darkblue')
if chart_5yr:
    chart_5yr.save('5_year_growth_chart_live.html')
    display(chart_5yr)

Calculating growth using current date: 2025-11-04
3-year start date: 2022-11-05
5-year start date: 2020-11-04



,Ticker,Growth_Pct
0,GOOGL,227.26
1,MSFT,149.19
2,UBER,143.28
3,META,123.27
4,AMZN,56.73
5,CRM,5.32
6,S&P 500,98.99


alt.Chart(...)

alt.Chart(...)